# Two AI workflows in one notebook

This lesson uses **ordinary Python cells** for a small pollinator survey. **Jupyter AI** can help you plan, review, or refactor through its separate chat/ACP agent. **nbinlineai** uses the tagged Markdown **AI questions** in this notebook to explain nearby work in place. Chat messages are not these cells, and neither extension automatically configures the other's provider.

To try both, install `nbinlineai` and `jupyter-ai` in the **same JupyterLab server environment** (for example, `uv add nbinlineai jupyter-ai` in a project), then restart the whole Jupyter server. Configure your own API key for nbinlineai in **Configure AI**. Jupyter AI's Claude/Codex ACP chat needs a separate adapter and authentication; follow [Jupyter AI's setup guide](https://jupyter-ai.readthedocs.io/en/stable/getting-started.html). No adapter, account, key, provider, model, or generated answer is bundled here. The Python analysis runs without Jupyter AI or an AI request.

Run the code cells one at a time first. Run an AI question only when you want an inline answer. **Run All Cells also runs new AI questions**, which may use your API provider; completed answers are protected by the notebook's default **Keep AI answers** setting.


In [ ]:
from dataclasses import dataclass
from statistics import mean


@dataclass(frozen=True)
class Survey:
    site: str
    habitat: str
    flowers: int
    pollinator_visits: int  # Each site was observed for ten minutes.


surveys: list[Survey] = [
    Survey("North meadow", "meadow", 20, 12),
    Survey("South meadow", "meadow", 18, 9),
    Survey("East courtyard", "courtyard", 24, 5),
    Survey("West courtyard", "courtyard", 16, 4),
]

for survey in surveys:
    print(survey.site, survey.pollinator_visits)


## Optional Jupyter AI chat: plan before editing

Open Jupyter AI's chat sidebar if you installed and authenticated an ACP agent. You can paste this request into **chat**; it is an instruction for the agent, not an nbinlineai AI question:

> Read the code cell with ID `jai-data-setup`. Propose one clear way to compare pollinator activity across habitats while accounting for different flower counts. Do not edit or execute cells yet. State two assumptions I should check.

If you want the agent to edit code later, review its proposed change first. This notebook already includes a working calculation below, so chat is optional.


In [ ]:
# Normalize visits to ten flowers, then compare the mean site rate by habitat.
rates_by_habitat: dict[str, list[float]] = {}
for survey in surveys:
    if survey.flowers <= 0:
        raise ValueError(f"{survey.site} needs a positive flower count")
    rate = 10 * survey.pollinator_visits / survey.flowers
    rates_by_habitat.setdefault(survey.habitat, []).append(rate)

mean_rates: dict[str, float] = {
    habitat: round(mean(rates), 2)
    for habitat, rates in rates_by_habitat.items()
}
for habitat, rate in sorted(mean_rates.items()):
    print(f"{habitat}: {rate:.2f} visits per ten flowers")


In [ ]:
# Small deterministic checks; they do not claim a causal effect.
assert len(surveys) == 4
assert all(survey.flowers > 0 for survey in surveys)
assert set(mean_rates) == {"courtyard", "meadow"}
assert mean_rates["meadow"] > mean_rates["courtyard"]
print("Four sites checked; meadow has the higher observed normalized rate.")


## nbinlineai question: explain the result in place

The next tagged Markdown cell is an **AI question**, not Jupyter AI chat. The notebook defaults to **Learning** style: it should guide your reasoning with questions rather than hand you a full solution. Click its **Run AI** control when ready. It will use code above as notebook context, within your chosen Context settings.


Why compare visits per ten flowers instead of raw visit counts in the code above? Ask me a guiding question about what the observed difference can and cannot establish.


## Optional Jupyter AI chat: refactor and validate

After reading the inline answer, you may use Jupyter AI chat for a separate coding task. Paste this into chat:

> Look at `jai-rate-analysis` and `jai-checks`. Suggest a small refactor into a named function that returns mean rates by habitat. Explain how to check that its output matches the existing calculation. Show a proposed patch, but do not apply or run it without my approval.

If you accept a change, rerun affected Python cells, then use nbinlineai's answer controls deliberately: a completed AI answer stays **kept** until you turn Keep off for that question. The Jupyter AI `run_cell` tool can execute a code cell by its stable ID, but in the default Jupyter AI setup it treats a tagged Markdown AI question as a no-op. Use nbinlineai's **Run AI** or native Shift+Enter for that question.


## Continue the Learning dialogue

Edit the next AI question to reflect **your own** interpretation before you run it. It is prefilled with a provisional thought, not a generated answer. The earlier inline question and its completed answer can inform this later turn. If you want a different answer to the earlier question after changing the analysis, turn off that cell's **Keep answer** first.


My provisional interpretation is that the meadow sites had a higher observed rate in this small sample, but the difference alone does not prove the habitat caused it. Ask me one follow-up question about sampling or other explanations.


## Optional: let nbinlineai add a code draft

This final step is separate from Jupyter AI chat. The code cell imports nbinlineai's special `insert_code` tool; it does not call a provider. The ordinary Markdown declaration then offers that tool to the last AI question. Run that question only if you want a new **unexecuted** code cell. The draft appears below its paired AI answer, remains editable, and is not executed in the current Run All batch. Review it before running it yourself, then save the notebook to keep it.


In [ ]:
from nbinlineai.tools import insert_code
print("insert_code is available for a later AI question; this cell makes no AI request.")


- &`insert_code` — add an editable, unexecuted code draft to this notebook.


Use `insert_code` to add a short Python cell that prints each habitat's number of surveyed sites from `rates_by_habitat`. Leave the new cell unexecuted so I can review it first.


## What to remember

Jupyter AI chat can help design or revise code, subject to its agent's own setup and permissions. nbinlineai questions stay in the notebook and can explain local code and earlier AI turns. Both can touch the same notebook: avoid simultaneous edits while an inline request is running. The [coexistence FAQ](https://rahuldave.com/nbinlineai/faq.html#can-i-run-nbinlineai-alongside-jupyter-ai-for-claude-or-codex-acp-chat) explains native Run All, individual Jupyter AI `run_cell`, API keys, and the limits of the compatibility test.
